In [24]:
def time_to_seconds(t):
    if pd.isna(t):
        return 0.0
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

In [25]:
import os
import pandas as pd
import subprocess

# === ПУТИ ===
CSV_PATH = "datas/train_soundscapes_labels.csv"
AUDIO_DIR = "datas/train_soundscapes"
OUTPUT_DIR = "chunks"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# === 1. Читаем CSV ===
df = pd.read_csv(CSV_PATH, dtype={"primary_label": str})

# === 2. Разбиваем источники ===
df["labels_list"] = df["primary_label"].apply(
    lambda x: x.split(";") if pd.notna(x) else []
)

# === 3. функция ffmpeg ===
def cut_audio(input_file, output_file, start, duration):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", input_file,
        "-ss", str(start),
        "-t", str(duration),
        "-c", "copy",
        output_file
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# === 4. Основной цикл ===
files = []
sources_in_files = []

for _, row in df.iterrows():
    filename = row["filename"]
    start = time_to_seconds(row["start"])
    end = time_to_seconds(row["end"])
    labels = row["labels_list"]

    input_path = os.path.join(AUDIO_DIR, filename)
    if not os.path.exists(input_path):
        continue

    duration = end - start

    # имя нового файла
    base = filename.replace(".ogg", "")
    chunk_name = f"{base}_{int(start)}_{int(end)}.ogg"
    output_path = os.path.join(OUTPUT_DIR, chunk_name)

    # === режем ===
    cut_audio(input_path, output_path, start, duration)

    # === сохраняем в списки ===
    # список звуковых файлов
    files.append(output_path)
    # список списков источников звука в файлах
    sources_in_files.append(labels)

# # === 5. Результат ===
print("\nКоличество 5 секундных эталонных файлов:", len(files))


Количество 5 секундных эталонных файлов: 1478


✅ список .ogg файлов
✅ для каждого файла — список источников (ID: буквенно-цифровые коды)
❗ неизвестны громкости и сдвиги

НАПРИМЕР

Преобразование ID → индексы

Алгоритму нужны числа, не строки.

In [26]:
def build_source_index(sources_in_files):
    unique_sources = sorted(set(s for lst in sources_in_files for s in lst))
    source_to_idx = {s: i for i, s in enumerate(unique_sources)}
    return source_to_idx, unique_sources

source_to_idx = {
    "A12": 0,
    "B7": 1,
    "C99": 2
}

👉 каждому источнику присваивается индекс

Построение A_mask

In [27]:
import numpy as np

def build_A_mask(sources_in_files, source_to_idx):
    M = len(sources_in_files)
    N = len(source_to_idx)

    A_mask = np.zeros((M, N), dtype=np.float32)

    for i, src_list in enumerate(sources_in_files):
        for s in src_list:
            j = source_to_idx[s]
            A_mask[i, j] = 1.0

    return A_mask

Пример результата
A_mask =
[
 [1, 1, 0],
 [0, 1, 1],
 [1, 0, 1],
]

1. Загрузка и STFT

In [28]:
import numpy as np
import librosa
import soundfile as sf
# загрузка ogg файлов
"""librosa.load(f, sr=32000, mono=True)
читает .ogg напрямую (через встроенный декодер)
важно:
автоматически декодирует OGG → PCM
приводит к частоте 32000 Гц
переводит в моно"""
def load_signals(files, sr=32000):
    signals = [librosa.load(f, sr=sr, mono=True)[0] for f in files]
    # обрезаем все сигналы до одной длины
    # STFT требует одинаковые размеры
    # иначе матрицы не сложатся
    min_len = min(len(s) for s in signals)
    signals = [s[:min_len] for s in signals]
    # M — число файлов
    # T — длина сигнала
    # signals.shape = (M, T)
    return np.array(signals), sr

# STFT — переход в частоты
# Каждый сигнал превращается в:
# (F, T) комплексная матрица
# где:
# F — частоты
# T — временные окна
def compute_stft(signals, n_fft=2048, hop=512):
    stfts = [librosa.stft(s, n_fft=n_fft, hop_length=hop) for s in signals]
    # X.shape = (F, T, M)
    # где:
    # M — количество файлов
    X = np.stack(stfts, axis=-1)  # (F, T, M)
    return X

2. Основной алгоритм (быстрый NMF с маской)

In [29]:
import numpy as np

def masked_nmf(V, A_mask, n_iter=80):
    """
    V: (F, T, M) — амплитуды спектров для M источников
    A_mask: (M, N) — маска (0/1), структура источников
    n_iter: число итераций
    """
    F, T, M = V.shape
    N = A_mask.shape[1]

    # --- инициализация ---
    W = np.abs(np.random.randn(F, N)) + 1e-6       # (F, N)
    H = np.abs(np.random.randn(N, T)) + 1e-6       # (N, T)
    G = (np.abs(np.random.randn(M, N)) + 1e-6) * A_mask  # (M, N)

    for _ in range(n_iter):
        # --- реконструкция WH ---
        WH = np.einsum('fn,nt->fnt', W, H)  # (F, N, T)

        # --- учитываем G ---
        # G двумерная (M, N) → расширяем по времени T для broadcast
        G_exp = G[:, :, np.newaxis]          # (M, N, 1)
        V_hat = np.einsum('mn,fnt->ftm', G, WH)  # (F, T, M)
        V_hat = np.maximum(V_hat, 1e-8)

        # --- обновление H ---
        num = np.einsum('ftm,mn,fn->nt', V / V_hat, G, W)
        # num: (N, T)
        # den: (N, T) → сумма по F и M для broadcast
        den = np.einsum('mn,fn->n', G, W)[:, np.newaxis]  # shape (N,1) → broadcast на T
        H *= num / den

        # --- обновление W ---
        num = np.einsum('ftm,mn,nt->fn', V / V_hat, G, H)
        # den = np.einsum('mn,nt->fn', G, H) + 1e-8
        # G: (M, N), H: (N, T)
        # хотим den: (F, N) → F ось не участвует в сумме, только N
        # den = np.sum(G * np.sum(H, axis=1, keepdims=True), axis=0)[np.newaxis, :] + #1e-8  # (1, N)
        den = np.einsum('mn,nt->n', G, H)[np.newaxis, :] + 1e-8  # shape (1, N)
        W *= num / den

        # --- обновление G ---
        num = np.einsum('ftm,fnt->mn', V / V_hat, WH)
        den = np.sum(WH, axis=(0,2), keepdims=True)  # shape (1, N, 1)
        den = np.tile(den, (M, 1, 1)).squeeze()      # shape (M, N)
        G *= (num / den) * A_mask  # соблюдаем структуру

    return W, H, G

3. Восстановление источников (ключевой момент)

In [30]:
def reconstruct_sources(X, W, H):
    """
    X: комплексный STFT (F, T, M)
    """
    F, T, M = X.shape
    N = H.shape[0]

    # спектры источников
    S_mag = np.einsum('fn,nt->fnt', W, H)  # (F, N, T)

    # soft mask
    total = np.sum(S_mag, axis=1, keepdims=True) + 1e-8
    masks = S_mag / total  # (F, N, T)

    # используем фазу первого канала
    phase = np.exp(1j * np.angle(X[:, :, 0]))

    sources = []
    for i in range(N):
        S_complex = masks[:, i, :] * np.abs(X[:, :, 0]) * phase
        s = librosa.istft(S_complex, hop_length=512)
        sources.append(s)
    print("\nКоличество одноисточниковых эталонных файлов ", len(sources))
    return sources

n_fft
n_fft = 2048  - лучше качество

🔹 hop_length
hop = 256 или 512

🔹 n_iter
30 → быстро
80 → качественно
4. Полный pipeline

In [31]:
def separate(files, A_mask):
    signals, sr = load_signals(files)
    X = compute_stft(signals)

    # V = np.abs(X)
    V = np.log1p(np.abs(X))

    W, H, G = masked_nmf(V, A_mask, n_iter=80)

    sources = reconstruct_sources(X, W, H)

    return sources, sr

5. Сохранение

In [32]:
def separate_pipeline(files, sources_in_files):
    signals, sr = load_signals(files)

    source_to_idx, idx_to_source = build_source_index(sources_in_files)
    A_mask = build_A_mask(sources_in_files, source_to_idx)

    X = compute_stft(signals)
    # V = np.abs(X)
    V = np.log1p(np.abs(X))

    W, H, G = masked_nmf(V, A_mask, n_iter=80)

    sources = reconstruct_sources(X, W, H)

    return sources, idx_to_source, sr

In [33]:
import soundfile as sf
from tqdm import tqdm
import os
sources, names, sr = separate_pipeline(files, sources_in_files)

output_dir = "alone_sources"
os.makedirs(output_dir, exist_ok=True)
# name это primary_label источника, а файл это звук источника
for s, name in tqdm(zip(sources, names), total=len(sources), desc="Saving alone_sources files", miniters=10):
    sf.write(os.path.join(output_dir, f"{name}.ogg"), s, sr)


MemoryError: Unable to allocate 3.49 GiB for an array with shape (1025, 309, 1478) and data type float64

Как улучшить ещё
🚀 1. Увеличить FFT
n_fft = 2048

→ лучше частотное разделение

🚀 2. Больше итераций
n_iter = 50–100
🚀 3. Лог-спектр
V = np.log1p(np.abs(X))
🚀 4. GPU (очень советую)

Могу переписать на PyTorch → ускорение ×10

🔴 Ограничения (честно)
если 2 источника очень похожи → сложнее
если один всегда доминирует → второй хуже восстанавливается